#**Laboratorio di Introduzione alla Matematica Computazionale - Esercizi in Julia**

##**Esercitazione 6: Interpolazione e fit di dati**

Le tecniche di interpolazione o *data fitting* si usano per risolvere problemi come il seguente.

Supponiamo di aver fissato dei *nodi di interpolazione*, cioè punti $\{x_i\}_{i=0,\dots ,n}$, con $x_{i}<x_{i+1}$, in un intervallo $[a,b]$ della retta reale, e di conoscere i valori $y_{i}=f(x_i)$ di una certa funzione $f$ (non nota) in corrispondenza dei nodi. Vogliamo approssimare i valori di $f$ in altri punti di $[a,b]$, i cosiddetti *query points*.

L'idea è di determinare una funzione $g$ che imiti $f$ in modo ``plausibile'' e in particolare assuma i valori prescritti sugli $x_i$, in modo esatto (interpolazione) o approssimato (fit). Una volta determinata $g$, la si valuta nei query points.

Per eseguire l'interpolazione in Julia useremo la libreria `Interpolations.jl`. Vediamo alcuni casi interessanti.

---
# Setup iniziale
Carichiamo le librerie necessarie.

In [ ]:
using Plots
using Interpolations
using CubicSplines
using Polynomials

Nota: se le istruzioni precedenti danno errore, potrebbe essere necessario caricare le librerie corrispondenti con le istruzioni
```julia
import Pkg; Pkg.add("Interpolations")
import Pkg; Pkg.add("Polynomials")
import Pkg; Pkg.add("CubicSplines")
```

---
# 1. Interpolazione


La tecnica di interpolazione più semplice è l'*interpolazione lineare*: prendiamo $g$ come la funzione lineare a tratti che collega i punti $(x_i,y_i).$

Per esempio, consideriamo la funzione $f(x)=1/(1+x^2)$ definita su $[-5,5]$ e supponiamo di conoscere i valori di $f$ nei nodi $x_i=5\cos(\frac{i\pi}{10})$ per $i=0,\ldots, 10$. Costruiamo questo esempio in Julia e disegnamo i punti da interpolare:

```julia
# Definiamo la funzione da interpolare
f(t) = 1 / (1 + t^2)

# Definiamo i nodi di interpolazione
n = 11
x = 5 .* cos.(range(0, stop=pi, length=n))
y = f.(x)

# Raffiguriamo i nodi
scatter(x, y, label="Punti noti", marker=:circle)
```

Ora cerchiamo un'approssimazione di $f$ in 100 punti equispaziati sull'intervallo $[-5,5]$, usando l'interpolazione lineare:

```julia
# 100 query points
interpx = range(-5, stop=5, length=100)

# Interpolazione lineare
# Nota: Interpolations.jl richiede nodi ordinati
p = sortperm(x)
nodes_x = x[p]
nodes_y = y[p]

itp_linear = linear_interpolation(nodes_x, nodes_y)
interpy_lin = itp_linear.(interpx)

# Raffiguriamo i risultati
plot(interpx, interpy_lin, label="lineare", color=:red)
scatter!(x, y, label="nodi", mc=:blue)
```

Un tipo di interpolazione molto diffuso è l'interpolazione con *spline cubiche*. In questo caso $g$ viene scelta come una funzione polinomiale a tratti, con polinomi di grado al più 3, e raccordi $C^2$ nei nodi. Proviamo ad applicare questo tipo di interpolazione al nostro esempio.

```julia
spline = CubicSpline(nodes_x, nodes_y)
interpy_cubic = spline[interpx]
```

Possiamo raffigurare tutti i risultati, aggiungendo anche al grafico i valori esatti della funzione.

```julia
# Raffiguriamo i risultati
scatter(x, y, label="nodi", mc=:blue)
plot!(interpx, interpy_lin, label="lineare", color=:red)
plot!(interpx, interpy_cubic, label="cubica", color=:green)
plot!(interpx, f.(interpx), label="esatta", color=:black, lw=2)
```

Un'altra tecnica per interpolare $n+1$ punti dati $(x_i,y_i)$ consiste nella scelta di $g$ come l'unico polinomio di grado $\leq n$ che assume i valori prescritti $y_i$ nei nodi $x_i$ (*interpolazione polinomiale*). Un modo conveniente per esprimere il polinomio interpolatore è dato dai *polinomi di Lagrange*.

Ricordiamo che i polinomi di Lagrange rispetto ai nodi $x_0,\ldots,x_n$ sono definiti come

$$
L_i(x)=\prod_{j=0,\ldots,n; j\neq i}\frac{x-x_j}{x_i-x_j},\qquad i=0,\ldots,n
$$

e formano una base dello spazio dei polinomi di grado al più $n$.

Si dimostra che il polinomio di interpolazione $p(x)$ si può scrivere nella base di Lagrange come
$$
p(x)=\sum_{i=0}^n y_i L_i(x).
$$


**Esercizio 1**. Scrivere una function `lagrange(x,y,interpx)` che prenda in ingresso
* il vettore `x` di lunghezza $n$ contenente i nodi di interpolazione;
* il vettore `y` di lunghezza $n$ contenente i valori da interpolare in corrispondenza dei nodi,
* il vettore `interpx` di lunghezza $m$ contenente i punti sui quali vogliamo valutare il polinomio interpolatore;

e restituisca in output il vettore `v` di lunghezza $m$ contenente i valori del polinomio interpolatore sui punti di `interpx`.

Suggerimento: valutate direttamente i polinomi di Lagrange su `interpx`, senza determinare il polinomio interpolatore in base monomiale.

**Esercizio 2**. Si usi la function `lagrange` appena definita per risolvere il problema dell'interpolazione per la funzione $f(x)=1/(1+x^2)$ su $[-5,5]$ sui punti $x_i=5\cos(\frac{i\pi}{10})$, $i=0,\ldots,10$, visti prima; si mostri l'errore di interpolazione disegnando la funzione e il polinomio interpolatore come nel grafico precedente.

Suggerimento: come *query points* prendere 100 punti linearmente equispaziati nell'intervallo $[-5,5]$.

**Esercizio 3**. Sempre per la funzione $f(x)=1/(1+x^2)$ definita su $[-5,5]$, eseguite e disegnate delle interpolazioni polinomiali di grado crescente, su nodi equispaziati. Che cosa osservate? L'approssimazione migliora al crescere del grado?

---
# 2. Fit di dati


Dati i punti $(x_i,y_i)$, $i=1,\ldots,m$ e un intero positivo $n<m$, il problema del fit polinomiale consiste nel determinare un polinomio $p(x)$ di grado $n$ tale che $p(x_i)\approx y_i$ nel senso dei minimi quadrati, cio\`e in modo da minimizzare la quantità
$$
(p(x_1)-y_1)^2+(p(x_2)-y_2)^2+\dots +(p(x_m)-y_m)^2.
$$
In Julia il calcolo dei coefficienti di $p(x)$ si pu\`o effettuare con il comando

```julia
fit(x,y,n)
```

dove `x` è il vettore degli $x_i$ e
`y`, il vettore degli $y_i$ ed `n` è il grado del polinomio.

**Esercizio 4**. Si consideri la funzione $f(x)=1/(x+(1-x)^2)$ sull'intervallo $[-2,2]$. Si valuti $f(x)$ in 20 punti equispaziati $x_i$, $i=1,\ldots, 20$, sull'intervallo $[-2,2]$ e si calcolino i coefficienti del polinomio $p(x)$ di grado 3 tale che $p(x_i)\approx f(x_i)$ nel senso dei minimi quadrati. Si tracci nella stessa figura il grafico di $f(x)$ e quello di $p(x)$. Si ripeta poi l'operazione per $n=5,8,10,17$, sempre nella stessa figura. Come si comporta la qualità di approssimazione della funzione all'aumentare del grado del polinomio?

**Esercizio 5**. Come nell'esercizio precedente, sia $f(x)=1/(x+(1-x)^2)$ definita sull'intervallo $[-2,2]$, e siano gli $x_i$ definiti come sopra. Consideriamo 100 punti $z_1,\ldots,z_{100}$ equidistanziati nel suddetto intervallo. Per le approssimazioni seguenti, valutare gli errori di approssimazione sugli $z_i$ e tracciarne il grafico:

* `fit` con grado $5$;

* `linear_interpolation` ;

* `CubicSpline`

* interpolazione polinomiale con la function `lagrange` dell'Esercizio 2.
